## Test Simple Sampling - sampler

In [1]:
%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.widb
%aimport quantdata.widb.client
%aimport quantdata.widb.day
%aimport quantdata.widb.sampler
%aimport quantdata.dysanos
%aimport quantdata.dysanos.data
%aimport quantdata.dysanos.data.simple

from quantdata import env#setup logging etc
from quantdata.dysanos.data.simple import SimpleSampler, SimpleConfig, SamplingConfig, ExpiryFilter, RawDayFilter, SampleDaysConfig, DateRange, PoolConfig
from cdxcore.verbose import Context
from IPython.display import display
from datetime import date

def test():
    simple_config = SimpleConfig(
            tickers = ['SPX','NDX','RUT'],
            date_range = DateRange(start=date(2025, 1, 1), end=date(2025, 7, 1)),
            raw_day_filter = RawDayFilter(),
            expiry_filter = ExpiryFilter(),
            sample_days_config = SampleDaysConfig(),
            sampling_config = SamplingConfig(num_samples=20)
    )
    pool_config = PoolConfig(num_workers=4, threading=False) # no parallel processing for testing
    verbose = Context.all
    refresh_cache = True

    sampler = SimpleSampler(simple_config=simple_config, pool_config=pool_config, refresh_cache=refresh_cache, verbose=verbose)
    return sampler

sampler = test()
print("/done")

00: Generating samples for 3 tickers NDX, RUT and SPX with from 2025-01-01 to 2025-07-01... 01:   Collecting enriched data for ticker NDX for 123 business days from 2025-01-02 to 2025-07-01 excluding half days.
03:       Sample Days Config: nopt=1000,bdte=756,nms=4.0,lvolu=1.0,loi=1.0,rmam=False,vegaSqrtVar=0.001
03:       Raw Day Filter: bid=0.011
03:       Expiry filter: eth=10,bth=4
03:       Pool config: workers 4p
01:   Regenerating all files.
01:   Generated all 123 missing business days for NDX using 4 processes. This took 7s.                                          
01:   Reading the generated enriched data for the 123 missing business days for NDX in the main process... done; this took 0.65s.
02:     The following 1 business days where removed because they had no data after filtering: 2025-01-08.
01:   Generating sample for 122 business days and at most 257 options per day for ticker NDX... done; this took 0.029s.
03:       +-------------------+------------+---------+--------

## Test sampling

In [2]:
"""
Shapes
"""

from prettytable import PrettyTable

def test_shapes():
    shapes = sampler.shapes( 10 )

    tbl = PrettyTable()
    tbl.field_names = ["Ticker", "Shape"]

    for k, v in shapes.items():
        tbl.add_row([k, v])
    
    print(tbl)

test_shapes()   



+-------------------+----------------+
|       Ticker      |     Shape      |
+-------------------+----------------+
|      is_valid     | (10, 25, 1000) |
|   ask_pure_delta  | (10, 25, 1000) |
|    ask_pure_dk    | (10, 25, 1000) |
|   ask_pure_gamma  | (10, 25, 1000) |
|   ask_pure_price  | (10, 25, 1000) |
|   ask_pure_theta  | (10, 25, 1000) |
|   ask_pure_vega   | (10, 25, 1000) |
|      ask_vol      | (10, 25, 1000) |
|   bid_pure_delta  | (10, 25, 1000) |
|    bid_pure_dk    | (10, 25, 1000) |
|   bid_pure_gamma  | (10, 25, 1000) |
|   bid_pure_price  | (10, 25, 1000) |
|   bid_pure_theta  | (10, 25, 1000) |
|   bid_pure_vega   | (10, 25, 1000) |
|      bid_vol      | (10, 25, 1000) |
|      call_ask     | (10, 25, 1000) |
|      call_bid     | (10, 25, 1000) |
|         df        | (10, 25, 1000) |
|        fwd        | (10, 25, 1000) |
|       is_am       | (10, 25, 1000) |
|    is_american    | (10, 25, 1000) |
| log_open_interest | (10, 25, 1000) |
|     log_volume    | (10

In [3]:
from prettytable import PrettyTable
from cdxcore.util import fmt_list, fmt_big_byte_number

def test_batch():
    print("Loading torch...", end='', flush=True)
    import torch as torch
    print("done.")
    g = torch.Generator(device="cpu")
    g.manual_seed(1234)
    batch_ix, batch = sampler(g, trn=True, batch_size=10, device="cpu", return_batch_indices=True)

    # indices
    tbl = PrettyTable()
    tbl.field_names = ["Ticker", "Date"]

    for bix in batch_ix:
        date = sampler.data[bix.ticker].dates[bix.index]
        tbl.add_row([bix.ticker, bix.date])
    print("Samples")
    print(tbl)

    tbl = PrettyTable()
    tbl.field_names = ["Ticker", "Shape", "Type", "Size", "Device"]

    for k, v in batch.items():
        tbl.add_row([k, tuple(v.shape), v.dtype, fmt_big_byte_number(v.nbytes), v.device])
    
    print(tbl)

test_batch()



Loading torch...done.
Samples
+--------+------------+
| Ticker |    Date    |
+--------+------------+
|  RUT   | 2025-01-31 |
|  RUT   | 2025-03-24 |
|  SPX   | 2025-01-07 |
|  RUT   | 2025-03-03 |
|  SPX   | 2025-03-03 |
|  NDX   | 2025-03-24 |
|  SPX   | 2025-01-24 |
|  SPX   | 2025-03-24 |
|  NDX   | 2025-03-31 |
|  SPX   | 2025-03-10 |
+--------+------------+
+-------------------+----------------+---------------+------------+--------+
|       Ticker      |     Shape      |      Type     |    Size    | Device |
+-------------------+----------------+---------------+------------+--------+
|      is_valid     | (10, 25, 1000) |   torch.bool  |  244.14KB  |  cpu   |
|   ask_pure_delta  | (10, 25, 1000) | torch.float32 |  976.56KB  |  cpu   |
|    ask_pure_dk    | (10, 25, 1000) | torch.float32 |  976.56KB  |  cpu   |
|   ask_pure_gamma  | (10, 25, 1000) | torch.float32 |  976.56KB  |  cpu   |
|   ask_pure_price  | (10, 25, 1000) | torch.float32 |  976.56KB  |  cpu   |
|   ask_pure_theta